# RADAR vs LiDAR: Recommendations and Conclusion

## Recommendations

### Q1. If the system requires stable full 2D vehicle velocity, should it primarily rely on RADAR or LiDAR?

**Recommendation**

Under the current pipeline, LiDAR should be used as the primary source of 2D vehicle velocity, while RADAR should be retained as an additional motion source and contribute to estimation or fusion when its estimate quality is sufficiently high.

**Evidence**

In the full-dataset paired comparison, the median speed error was 0.443 m/s for LiDAR and 0.572 m/s for RADAR. The median difference was relatively small, but RADAR showed a much larger tail error, with a P90 of 30.469 m/s compared with 2.071 m/s for LiDAR. This indicates that the main weakness of the current RADAR 2D velocity output is its heavy-tail behaviour, while LiDAR provides more consistent estimates.

RADAR 2D reconstructability was approximately 47.3%, which was close to the 46.9% LiDAR velocity availability. RADAR dropped to 18.5% only after applying the current stability criterion. The current results therefore support using LiDAR as the more reliable primary 2D velocity source, while retaining RADAR as valuable complementary motion information.

**Methodological Clarification**

The condition-based analysis in this project evaluates the final reconstructed RADAR 2D velocity produced by the current pipeline, rather than the intrinsic accuracy of the native RADAR Doppler measurement.

Therefore, the Area, Weather, Visibility, Distance and Object Type results should be interpreted as:

> How the reliability of the current RADAR 2D velocity reconstruction changes under different operating conditions.

A supplementary native RADAR validation on the trainval subset found a low median radial-velocity error (0.22 m/s), but also a heavy tail (P90 approximately 25 m/s). This suggests that part of the observed 2D error may arise during the reconstruction stage, but some large errors already exist in the radial measurement itself.

This supplementary validation is diagnostic, as radial-velocity error and full-2D speed error are not directly equivalent metrics.

**Why is 2D velocity used instead of 3D velocity?**

This project focuses on road-vehicle motion, so 2D ground-plane velocity directly represents the longitudinal and lateral motion most relevant to the driving scenarios being analysed.

The annotation reference already contains $v_x$, $v_y$, $v_z$, together with both 2D and 3D speed. However, in normal road-driving scenarios, vertical velocity is mainly associated with road elevation or slope rather than the vehicle's primary motion behaviour.

Therefore:

> 2D ground-plane velocity is used as the primary benchmark because it best represents the road-vehicle motion of interest in this project. 3D velocity may provide additional value in elevation-specific scenarios, such as ramps or steep roads, but it is not the primary evaluation target here.

### Q2. How should RADAR and LiDAR be used across different operating areas?

**Recommendation**

A fixed sensor strategy should not be used across all operating areas. In City and Terminal environments, RADAR can contribute more actively when its estimate is stable. In Highway and Rural environments, greater reliance should be placed on LiDAR, with stricter quality checks applied to RADAR outputs.

**Evidence**

Within the paired stable samples, RADAR showed median speed errors comparable to, and in some cases lower than, LiDAR in City and Terminal environments. In contrast, Highway and Rural environments showed both higher median errors and substantially larger P90 errors for RADAR, indicating poorer robustness.

Therefore, Area should be treated as a contextual factor for sensor confidence rather than a hard rule for selecting one sensor.

As this is a condition-based observational analysis, Area may also reflect differences in distance, speed and object distribution, so the observed performance differences should not be attributed to Area alone.

The interaction analysis supports this pattern: relative RADAR error increased significantly in Highway and Rural environments compared with City, indicating a larger RADAR–LiDAR performance gap in these areas.

### Q3. How should RADAR and LiDAR be used under different weather conditions?

**Recommendation**

Weather affects coverage and velocity-estimation reliability differently. In Fog and Snow, RADAR provides greater velocity-information coverage, but this should not be interpreted as consistently better 2D velocity accuracy. Weather should therefore influence sensor confidence rather than determine sensor selection by itself.

**Evidence**

In the full dataset, RADAR reconstructability was approximately 51.1% in Fog, compared with 31.1% for LiDAR availability, and approximately 50.6% vs 40.0% in Snow.

Under Clear conditions, the two sensors showed almost identical availability, while LiDAR was slightly higher in Rain and Overcast.

Therefore:

> RADAR provides the clearest complementary coverage advantage in Fog and Snow, while the difference between the two sensors is smaller under other weather conditions.

The interaction analysis shows that Fog increased the relative RADAR–LiDAR error gap compared with Clear conditions, while Rain, Snow and Overcast reduced that gap. This confirms that greater RADAR coverage does not necessarily correspond to better reconstructed 2D velocity accuracy.

### Q4. How do RADAR and LiDAR compare when object visibility is reduced or the target is partially occluded?

**Recommendation**

When object visibility is low, both sensors lose availability. RADAR retains additional motion coverage for some heavily occluded targets, but low visibility alone does not establish a clear accuracy advantage for either sensor. Visibility is therefore better used as an input to sensor confidence.

**Evidence**

As object visibility decreases, both RADAR and LiDAR velocity availability drop substantially. At Visibility Level 1, RADAR reconstructability remains approximately 25.5%, compared with 20.0% LiDAR availability, showing that RADAR can preserve additional motion information for some poorly visible or occluded targets.

### Q5. How should sensor use change as vehicles get farther away?

**Recommendation**

At short range, both sensors can contribute effectively to velocity estimation. Between 25–75 m, LiDAR is the stronger source of full 2D velocity. Beyond approximately 75 m, LiDAR coverage decreases more rapidly, and RADAR becomes increasingly valuable for maintaining long-range motion information. Beyond 100 m, the current data is too sparse to justify selecting one sensor solely by distance, so available information should be retained and quality-checked.

**Evidence**

At 0–25 m, RADAR shows a slightly lower median error, while LiDAR has substantially better P90 robustness.

Between 25–75 m, LiDAR provides clearly lower median and tail errors, making it more suitable for full 2D velocity estimation.

Beyond 75 m, RADAR reconstructability exceeds LiDAR availability. At 100–125 m, for example, RADAR reconstructability is approximately 34.1%, compared with 18.4% for LiDAR, although the stable RADAR 2D rate falls to approximately 0.9%.

The interaction analysis further shows a significant positive Sensor × Distance effect, indicating that the relative error of reconstructed RADAR 2D velocity increases with distance compared with LiDAR.

> As range increases, RADAR becomes increasingly valuable for maintaining motion coverage, but its reconstructed full-2D velocity becomes less reliable relative to LiDAR and should therefore be used more selectively.

### Q6. Should RADAR and LiDAR be used differently for different vehicle types?

**Recommendation**

Vehicle type can be used to adjust sensor confidence, but it should not be used as a direct sensor-switching rule. LiDAR is more consistent for common classes such as Cars and Trucks, while RADAR can be competitive for classes such as Buses, Construction Vehicles and Trains when a stable estimate is available.

**Evidence**

Within paired stable samples, RADAR shows median speed errors comparable to or lower than LiDAR for several vehicle classes, including Buses, Construction Vehicles and Trains.

However, for the most common classes such as Cars and Trucks, RADAR shows substantially larger tail errors, indicating poorer robustness. A similar pattern appears for Trailers, where typical error can be competitive but extreme errors remain much larger.

Object type should therefore be treated as an additional confidence factor rather than a hard rule for selecting one sensor.

Rare classes such as Ambulances and Police vehicles do not currently have enough samples to support strong recommendations.

The interaction analysis also confirms that the relative RADAR–LiDAR performance varies by vehicle class, supporting the use of object type as an additional confidence factor.

### Q7. When should RADAR and LiDAR be fused?

**Recommendation**

The current results do not support simple fusion as the default approach. Fusion should only be applied when both sensor estimates pass quality checks and are mutually consistent; otherwise, the higher-quality single-sensor estimate should be retained.

**Evidence**

The current simple fusion achieves a median speed error of approximately 0.556 m/s and a P90 of approximately 21.70 m/s, and does not outperform the best single-sensor result.

Therefore, the preferred approach is selective fusion rather than unconditional fusion.

### Q8. Which sensor is better for detecting objects, and can a pretrained model be used directly on truck data?

**Recommendation**

LiDAR should be used as the main detector. The pretrained model cannot be used on truck RADAR data as it is and would need truck-specific fine-tuning first. RADAR detection is best seen as a small safety net that catches a few objects LiDAR misses.

**Evidence**

Both sensors were tested with the same OmniHD-pretrained PointPillars model, on the same 10,063 samples, the same 87,781 ground-truth boxes, the same range (about 60 m × 40 m) and the same matching rule. This makes it a fair comparison.

- mAP: LiDAR 17.5%, RADAR 0.63%
- Precision: LiDAR 80.2%, RADAR 36.0%
- Recall: LiDAR 39.1%, RADAR 3.9%
- Car AP: LiDAR 66.5%, RADAR 2.3%
- Centre error when an object was found: LiDAR 0.44 m, RADAR 1.02 m
- Recall on moving objects: LiDAR 45.0%, RADAR 5.1%
- Recall on stationary objects: LiDAR 20.9%, RADAR 0.02%
- Recall in fog: LiDAR 44.0%, RADAR 1.0%

Pedestrians and riders were effectively not detected by either sensor.

**Does RADAR add anything?**

A little. Using both sensors raised recall from 39.1% (LiDAR alone) to 40.2%. Of the objects LiDAR missed, RADAR found 1.9%, rising to 3.5% at 50–75 m and 3.8% for large vehicles.

### Q9. Why is the RADAR score so low?

The gap is real, but it comes from a mix of reasons:

- **Physics.** RADAR has few points per object and finds objects mainly through their motion. This is why it almost never detects stationary objects (0.02%).
- **The model was trained on cars, not trucks.** It was trained on OmniHD passenger-car data and applied to trucks without retraining. Truck RADARs are mounted higher and arranged differently (for example, there is no centre-front RADAR), so RADAR data changes more between the two datasets than LiDAR data does.
- **The scoring rule is strict.** A detection only counts if its centre is within 1 to 2 m of the real object. RADAR places boxes less precisely, so it fails this rule more often, even when it has sensed the object.

**Caveats**

- This result should be read as a lower bound for RADAR under untuned conditions, not proof that RADAR cannot detect objects.
- Detection was only evaluated within about 60 m, so it says nothing about long range.
- The detection pattern differs from the velocity results in Q2 and Q3. For example, RADAR detection was best on highways and in clear weather, and showed no advantage in fog. Detection and velocity should therefore be reported separately rather than combined into one sensor ranking.

### Q10. What should be checked or improved before using RADAR in a real system?

**Recommendation**

A deployed system should include RADAR quality checks, and several parts of the current setup should be improved before drawing final conclusions about RADAR.

**Evidence**

Data quality issues found when inspecting the mini dataset:

- RADAR runs at about 19.5 Hz, but some scans are missing, with gaps of up to about 150 ms.
- Each scan contains at most 800 points.
- About 0.03% of RADAR points have impossible velocities of up to ±50 m/s and should be filtered out.
- Timestamps between sensors should be checked so each RADAR scan is matched with the correct truck motion.

Improvements to the current setup:

- Correct for the truck's own motion more carefully. The current velocity pipeline uses the cabin motion sensor only and does not account for the truck turning. In one test sample of a turning truck, this left only 43.5% of stationary objects reading as stationary, compared with 73% using a different correction source. This is a possible cause of RADAR's large velocity errors and needs testing on more data.
- Fine-tune the detection model on truck data, especially for RADAR (see Q8).
- Use a fairer availability rule. LiDAR is counted whenever it gives an estimate, but RADAR only when it is stable, which makes RADAR look worse on coverage.
- Run the analysis on the full dataset. This project used sets 1, 3 and 5 only, which is 820,213 of 2,586,264 sample_data records (about 32%).

### Q11. Which sensor should be trusted for a vehicle's direction of travel?

**Recommendation**

LiDAR should be used as the primary source for direction of travel. RADAR direction estimates should only be used after a consistency check, as they occasionally point the wrong way entirely.

**Evidence**

On matched vehicles in the mini dataset, LiDAR's median direction error was 0.94° in City and 0.44° on Highway, compared with 5.06° and 3.76° for RADAR. The difference was largest in the tail. On the highway, 10% of RADAR estimates were more than 146° from the true direction, so they were close to the opposite, whereas LiDAR's worst 10% stayed below 1.5°.

Direction is just as crucial as speed for predicting where a vehicle will go. A correct speed with a reversed heading is a serious error for path planning.

**Caveats**

The results come from the mini dataset (10 scenes). The Terminal result rests on only 14 moving vehicles and therefore should not be interpreted. Direction was only evaluated for vehicles moving at 2 m/s or faster.

### Q12. How should RADAR velocity be used in a real system?

**Recommendation**

A real system should use RADAR's radial velocity directly as a measurement inside a tracker, such as a Kalman filter, as opposed to reconstructing a full 2D velocity from each pair of frames. RADAR's higher update rate should also be used, which the current keyframe-based pipeline does not do.

**Evidence**

Most targets receive very few RADAR returns, usually from a single sensor. In the mini dataset:

- A target had a median of 3 associated returns.
- 2,845 of 9,588 target observations had only one return.
- The median number of RADAR sensors observing a target was 1.

Reconstructing a 2D velocity requires returns from clearly different viewing angles. Most targets do not provide this. In the mini dataset, 48.95% of intervals could be reconstructed, but only 19.16% passed the stability check.

The radial measurement itself is more stable than the reconstruction suggests. Its median error was 0.22 m/s on the trainval subset. A tracker can combine single radial measurements over time and does not need several viewing angles in the same frame.

RADAR runs at around 19.5 Hz and LiDAR at around 10 Hz, however, every analysis in this project uses keyframes at 2 Hz. RADAR's main timing advantage has therefore not yet been tested.

**Caveats**

This is a recommendation for system design, not something which was tested in this project. The radial measurements also display a heavy error tail (90th percentile about 25 m/s), therefore, a tracker would still need outlier rejection.

### Q13. How should sensor confidence change with the target vehicle's speed?

**Recommendation**

Target speed should be treated as a confidence factor alongside distance and area. Both sensors are reliable for slow and stationary targets. At medium and high speeds, RADAR availability drops and its error grows faster than LiDAR's.

Results should also be reported by speed band, as the overall figures are dominated by slow vehicles.

**Evidence**

In the mini dataset, 43.6% of vehicle intervals were below 10 km/h, therefore, overall medians mostly describe slow traffic.

| Speed band | Stable RADAR availability | LiDAR availability |
|---|---|---|
| 0–10 km/h | 23.2% | 57.6% |
| 40–50 km/h | 7.3% | 27.8% |

On matched intervals, the two sensors had similar median errors at low speed. RADAR's median error became larger and more variable in several medium- and high-speed bands. LiDAR's median error also rose, from 0.18 m/s below 10 km/h to about 0.46 m/s between 60 and 90 km/h, but less sharply.

**Caveats**

The relationship was not consistent from one speed band to the next. Speed is also linked to Area, as highway traffic is faster, so the two effects overlap.

### Q14. How important is timing between sensors?

**Recommendation**

A deployed system should correct for timing differences between sensors and within each LiDAR scan. At highway speeds, the timing offsets in this dataset are large enough to move a vehicle by more than a metre.

**Evidence**

In the mini dataset:

- RADAR keyframes were up to 55.8 ms away from their annotation time (median 5.4 ms).
- LiDAR keyframes were up to 49.5 ms away (median 19.0 ms).
- The six LiDAR sensors were not synchronised with each other, and a single LiDAR scan spanned several tens of milliseconds.

At 110 km/h (about 30.6 m/s), a 55 ms offset corresponds to around 1.7 m of vehicle movement.

This timing spread is the likely reason that counting LiDAR points inside the annotation box gave 519 points for one test vehicle, whereas the dataset reported 430. The project therefore used the dataset's own point counts. A real system is not able to do that, so it would need to correct each LiDAR point for motion using its timestamp.

**Caveats**

The point count mismatch was examined for a single annotation. Its effect on the velocity results has not been measured.

### Q15. How should velocity performance be judged for a safety-relevant system?

**Recommendation**

Sensor performance should be judged against a stated accuracy requirement, with emphasis on how often large errors occur, as opposed to median error alone.

**Evidence**

Median errors make the two sensors look similar: 0.443 m/s for LiDAR and 0.572 m/s for RADAR on the paired trainval comparison. The 90th-percentile errors are very different: 2.071 m/s for LiDAR and 30.469 m/s for RADAR.

Simple fusion also seems reasonable by median (0.556 m/s), however, it has a 90th-percentile error of 21.70 m/s.

A rare large error is what matters for safety, and a median does not show it. Reporting metrics such as the following would make the comparison clearer for system design:

- The share of estimates within ±1 m/s.
- The share of estimates wrong by more than 5 m/s.

**Caveats**

The appropriate accuracy threshold depends on the intended system and should be set with the client.

## Conclusion

The main question of this project was how RADAR and LiDAR compare for autonomous trucking. Based on the MAN TruckScenes data we analysed (trainval sets 1, 3 and 5), neither sensor is simply better than the other. LiDAR is the more reliable sensor overall, and RADAR is a useful complement in specific situations.

**LiDAR is the stronger primary sensor.** For velocity, LiDAR's typical error was similar to RADAR's (median 0.443 vs 0.572 m/s), but LiDAR was much more consistent. Its P90 error was 2.071 m/s, compared with 30.469 m/s for RADAR (Q1). LiDAR was also far more reliable for direction of travel. On the highway in the mini dataset, 10% of RADAR direction estimates were more than 146° off, while LiDAR's worst 10% stayed below 1.5° (Q11). For object detection, LiDAR clearly outperformed RADAR on the same test (mAP 17.5% vs 0.63%, recall 39.1% vs 3.9%), and placed objects more precisely (centre error 0.44 m vs 1.02 m) (Q8).

**RADAR still adds value in some situations.** RADAR kept more raw motion information than LiDAR in fog and snow, for partly hidden objects, and beyond 75 m, where LiDAR coverage drops (Q3 to Q5). It also did well in slower areas such as city and terminal, and on classes such as buses, construction vehicles and trains (Q2, Q6). In detection, RADAR found 1.9% of the objects LiDAR missed, rising to 3.5% at 50–75 m and 3.8% for large vehicles (Q8). However, in most of these situations only a small share of RADAR estimates were stable enough to trust. For example, at 100–125 m only 0.9% of RADAR 2D estimates were stable.

| | LiDAR | RADAR |
|---|---|---|
| Main strength | Consistent velocity, reliable direction and precise object position | Senses motion directly, runs at a higher rate, keeps more raw information in fog, snow and at long range |
| Main weakness | Coverage drops in fog and at long range | Occasional very large speed and direction errors, few points per object, almost no detection of stationary objects |
| Recommended role | Primary sensor | Quality-checked complement |

**The two sensors work best together, but not by simple averaging.** The truck carries both sensors, and each one covers some of the other's gaps. However, the simple fusion method we tested was not better than LiDAR alone, because RADAR's occasional large errors pulled LiDAR's good estimates down (Q7). The recommended approach is selective fusion: LiDAR as the backbone, with RADAR used only when its estimate passes a quality check.

**RADAR's low numbers do not mean RADAR is a bad sensor.** Part of the gap comes from physics. RADAR returns far fewer points per object and mainly picks up moving objects. Part of it comes from our setup (Q9, Q10). The detection model was trained on passenger cars and was not adapted to trucks. The scoring rule is strict about position. The correction for the truck's own motion does not yet account for turning. Our pipeline also rebuilds a full 2D velocity from each pair of keyframes at 2 Hz, so it does not use RADAR's 19.5 Hz update rate or feed its radial velocity directly into a tracker (Q12). The RADAR results should therefore be read as a lower bound under untuned conditions.

**Final recommendation.** We recommend LiDAR as the primary sensor for both object detection and velocity. RADAR should be used as a quality-checked complement for bad weather, long range and large vehicles, ideally as a radial-velocity input to a tracker rather than a stand-alone 2D velocity. Sensor confidence should be adjusted by distance, area, weather, visibility, vehicle type and target speed, rather than switching sensors on one factor alone (Q2 to Q6, Q13). Any system built on these findings should be judged by how often large errors occur against an accuracy limit agreed with the client, not by median error alone (Q15). Before drawing final conclusions about RADAR, the improvements in Q10 and Q14 should be completed: truck-specific model training, better motion and timing correction, a fairer availability rule, and analysis of the full dataset.